# 08 — InceptionV3 pipeline on Colab (CUDA GPU)
Runs the full MILAN Section-7 pipeline for InceptionV3 on a real CUDA GPU, which is dramatically faster than local MPS for the heavy stages (exemplars + the per-unit importance loop in the edit).

**Before running:** `Runtime ▸ Change runtime type ▸ GPU` (T4 is fine; A100/L4 faster).

What this notebook does: clone the repo, install deps (keeping Colab's CUDA torch), rebuild the dataset deterministically, obtain the trained checkpoint, then run exemplars → descriptions → identify text neurons → editing, all on `--device cuda`. See [`docs/inception_v3_experiment.md`](https://github.com/1minute99/DSC291_Research_Reproduction/blob/main/docs/inception_v3_experiment.md) for what each stage means.

In [ ]:
# 0. Confirm we have a GPU.
!nvidia-smi

In [ ]:
# 1. Clone the repo + the vendored MILAN submodule.
#    Point BRANCH at whatever branch your Inception work is pushed to.
REPO_URL = 'https://github.com/1minute99/DSC291_Research_Reproduction.git'
BRANCH   = 'inception-extension'   # <-- the branch you pushed from your Mac

import os
if not os.path.isdir('/content/DSC291_Research_Reproduction'):
    !git clone --recurse-submodules --branch {BRANCH} {REPO_URL} /content/DSC291_Research_Reproduction
%cd /content/DSC291_Research_Reproduction
!git submodule update --init --recursive
!git log --oneline -1

In [ ]:
# 2. Install deps. KEEP Colab's preinstalled CUDA torch/torchvision (don't let
#    milan/requirements.in's CPU pins override them), and SKIP allennlp -- the
#    repo's _shims/allennlp handles it (see milan_glue/upstream.py).
!grep -ivE '^[[:space:]]*(allennlp|torch|torchvision)([[:space:]<>=!]|$)' milan/requirements.in > /tmp/req_milan_colab.txt
!pip install -q -r /tmp/req_milan_colab.txt
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

In [ ]:
# 3. Environment variables + import paths (mirror environment.md).
import os, sys
REPO = '/content/DSC291_Research_Reproduction'
os.environ['MILAN_DATA_DIR']   = f'{REPO}/data'
os.environ['MILAN_MODELS_DIR'] = f'{REPO}/models'
os.environ['MILAN_RESULTS_DIR']= f'{REPO}/results'
os.environ['PYTHONPATH']       = f'{REPO}:{REPO}/milan'   # for `!python -m ...`
for p in (REPO, f'{REPO}/milan'):
    if p not in sys.path:
        sys.path.insert(0, p)
print('env set')

In [ ]:
# 4. Rebuild the spurious dataset. Deterministic (seed=0), so this regenerates
#    the IDENTICAL dataset you built locally -- no need to upload 13k images.
#    Downloads Imagenette (~325 MB) on first run.
!python -m milan_repro.data.build_splits

## 5. Get the trained InceptionV3 checkpoint
Pick **one** of the two cells below.

- **5a (recommended for continuity):** reuse your exact `inception_v3_spurious.pth` (the patience=12 model you analysed). Upload it to Google Drive first (e.g. `MyDrive/dsc291/inception_v3_spurious.pth`), then run 5a to copy it in.
- **5b (fully self-contained):** retrain on the GPU. Fast (~minutes), but produces a *different* model than your MPS run, so accuracies will shift slightly.

In [ ]:
# 5a. Reuse the exact checkpoint from Google Drive.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_CKPT = '/content/drive/MyDrive/dsc291/inception_v3_spurious.pth'  # <-- adjust
!mkdir -p models && cp "{DRIVE_CKPT}" models/inception_v3_spurious.pth
!ls -lh models/inception_v3_spurious.pth

In [ ]:
# 5b. (Alternative) Retrain on the GPU instead of uploading.
# !python -m milan_repro.train.train_inception_v3 --device cuda

In [ ]:
# 6. Exemplars (dissection): top-k activating regions for all ~3,936 units at 299px.
!python -m milan_repro.milan_glue.run_exemplars --arch inception_v3 --device cuda

In [ ]:
# 7. Descriptions: MILAN `base` decoder captions each unit (downloads ~1 GB decoder
#    + ResNet101 encoder weights on first run).
!python -m milan_repro.milan_glue.run_descriptions \
    --dissect-dir results/edit/imagenet-spurious-text/inception_v3_spurious-50pct \
    --out results/descriptions_inception_v3.csv --device cuda

In [ ]:
# 8. Flag text neurons (description contains text/word/letter).
!python -m milan_repro.editing.identify_text_neurons \
    --descriptions results/descriptions_inception_v3.csv \
    --out results/descriptions_inception_v3_annotated.csv

In [ ]:
# 9. Editing experiment: per-unit importance + text-sorted / sort-all / random
#    ablation curves. This is the stage that took ~31 h on MPS; on a CUDA GPU
#    it is far faster. Raise --ablation-step / lower --ablation-max to trim more.
!python -m milan_repro.editing.evaluate --arch inception_v3 --device cuda

In [ ]:
# 10. Figures + summary.
!python -m milan_repro.figures.plot_fig7 \
    --dissect-dir results/edit/imagenet-spurious-text/inception_v3_spurious-50pct \
    --descriptions results/descriptions_inception_v3_annotated.csv \
    --out results/figs/fig7_inception_v3.pdf
!python -m milan_repro.figures.plot_fig8 \
    --ablation-csv results/ablation_curve_inception_v3.csv \
    --out results/figs/fig8_inception_v3.pdf

import pandas as pd
df = pd.read_csv('results/ablation_curve_inception_v3.csv')
base = df[df['mode']=='baseline'].iloc[0]['adv_acc']
best = df[df['mode'].isin(['text-sorted','sort-all'])].groupby('mode')['adv_acc'].max()
print('adv baseline:', round(base,4))
print('text-sorted recovery: +{:.1f} pt'.format(100*(best.get('text-sorted',float('nan'))-base)))
print('sort-all   recovery: +{:.1f} pt'.format(100*(best.get('sort-all',float('nan'))-base)))

In [ ]:
# 11. Persist results back to Drive so they survive the Colab session.
!mkdir -p /content/drive/MyDrive/dsc291/results_inception
!cp -r results/descriptions_inception_v3*.csv results/ablation_curve_inception_v3.csv results/importance_inception_v3.csv results/figs /content/drive/MyDrive/dsc291/results_inception/ 2>/dev/null
!ls -R /content/drive/MyDrive/dsc291/results_inception